Just merging all the years together to make a working set.

In [2]:
import pandas as pd
import os

folder_path = r"C:\Users\priva\OneDrive\Documents\Honors Project\Data\Medicare"
save_path = r"C:\Users\priva\OneDrive\Documents\Honors Project\Data"

all_dfs = []

for year in range(2013, 2024):
    file_path = os.path.join(folder_path, f"{year} medicare.xlsx")
    if os.path.exists(file_path):
        df = pd.read_excel(file_path)
        df["Year"] = year
        all_dfs.append(df)

combined_df = pd.concat(all_dfs, ignore_index=True)

working_file = os.path.join(save_path, "medicare_working_dataset.csv")
combined_df.to_csv(working_file, index=False)

sample_df = combined_df.sample(n=5000, random_state=1)
sample_file = os.path.join(save_path, "medicare_sample_dataset.csv")
sample_df.to_csv(sample_file, index=False)




In [3]:
import pandas as pd
import os

# paths
save_path = r"C:\Users\priva\OneDrive\Documents\Honors Project\Data"
working_file = os.path.join(save_path, "medicare_working_dataset.csv")

# load working dataset
df = pd.read_csv(working_file)

# quick structure checks
print(df.shape)
print(df.columns.tolist())

# basic missingness (top 15)
print((df.isna().mean().sort_values(ascending=False).head(15) * 100).round(2))

# key counts
print("years:", df["Year"].nunique())
print("hospitals (CCN):", df["Rndrng_Prvdr_CCN"].nunique())
print("drgs:", df["DRG_Cd"].nunique())
print("states:", df["Rndrng_Prvdr_State_Abrvtn"].nunique())

# outcome summaries
print(df["Avg_Submtd_Cvrd_Chrg"].describe())
print(df["Avg_Tot_Pymt_Amt"].describe())
print(df["Avg_Mdcr_Pymt_Amt"].describe())

# check duplicates on hospital x drg x year
k = ["Rndrng_Prvdr_CCN", "DRG_Cd", "Year"]
print("duplicate hospital-drg-year rows:", df.duplicated(k).sum())

# top DRGs by hospital coverage
top_drgs = (
    df.groupby("DRG_Desc")["Rndrng_Prvdr_CCN"]
      .nunique()
      .sort_values(ascending=False)
      .head(25)
)
print(top_drgs)

# top states by row count
top_states = df["Rndrng_Prvdr_State_Abrvtn"].value_counts().head(20)
print(top_states)


(1985253, 16)
['Rndrng_Prvdr_CCN', 'Rndrng_Prvdr_Org_Name', 'Rndrng_Prvdr_City', 'Rndrng_Prvdr_St', 'Rndrng_Prvdr_State_FIPS', 'Rndrng_Prvdr_Zip5', 'Rndrng_Prvdr_State_Abrvtn', 'Rndrng_Prvdr_RUCA', 'Rndrng_Prvdr_RUCA_Desc', 'DRG_Cd', 'DRG_Desc', 'Tot_Dschrgs', 'Avg_Submtd_Cvrd_Chrg', 'Avg_Tot_Pymt_Amt', 'Avg_Mdcr_Pymt_Amt', 'Year']
Rndrng_Prvdr_RUCA            0.05
Rndrng_Prvdr_RUCA_Desc       0.05
Rndrng_Prvdr_City            0.00
Rndrng_Prvdr_CCN             0.00
Rndrng_Prvdr_St              0.00
Rndrng_Prvdr_State_FIPS      0.00
Rndrng_Prvdr_Zip5            0.00
Rndrng_Prvdr_Org_Name        0.00
Rndrng_Prvdr_State_Abrvtn    0.00
DRG_Cd                       0.00
DRG_Desc                     0.00
Tot_Dschrgs                  0.00
Avg_Submtd_Cvrd_Chrg         0.00
Avg_Tot_Pymt_Amt             0.00
Avg_Mdcr_Pymt_Amt            0.00
dtype: float64
years: 11
hospitals (CCN): 3497
drgs: 684
states: 51
count    1.985253e+06
mean     6.680922e+04
std      8.522564e+04
min      1.281417e+03


In [4]:
import numpy as np

# keep only needed cols and fix types
num_cols = ["Tot_Dschrgs", "Avg_Submtd_Cvrd_Chrg", "Avg_Tot_Pymt_Amt", "Avg_Mdcr_Pymt_Amt"]
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")

# drop obvious bad rows
df = df[df["Avg_Submtd_Cvrd_Chrg"] > 0].copy()
df = df[df["Tot_Dschrgs"] > 0].copy()

# add logs (main outcomes)
df["log_charge"] = np.log(df["Avg_Submtd_Cvrd_Chrg"])
df["log_tot_pay"] = np.log(df["Avg_Tot_Pymt_Amt"].clip(lower=1))
df["log_mdcr_pay"] = np.log(df["Avg_Mdcr_Pymt_Amt"].clip(lower=1))

# quick ratio features
df["charge_to_totpay"] = df["Avg_Submtd_Cvrd_Chrg"] / df["Avg_Tot_Pymt_Amt"].replace(0, np.nan)
df["totpay_to_mdcr"] = df["Avg_Tot_Pymt_Amt"] / df["Avg_Mdcr_Pymt_Amt"].replace(0, np.nan)

print(df.shape)
print(df[["log_charge","log_tot_pay","log_mdcr_pay","charge_to_totpay","totpay_to_mdcr"]].describe())

# biggest charge outliers (overall)
outliers = df.sort_values("Avg_Submtd_Cvrd_Chrg", ascending=False).head(20)
print(outliers[["Year","Rndrng_Prvdr_State_Abrvtn","Rndrng_Prvdr_CCN","DRG_Cd","DRG_Desc","Tot_Dschrgs","Avg_Submtd_Cvrd_Chrg","Avg_Tot_Pymt_Amt","Avg_Mdcr_Pymt_Amt"]])

# max/min ratio by DRG (variation scan)
var = df.groupby("DRG_Desc")["Avg_Submtd_Cvrd_Chrg"].agg(["min","median","max","count"])
var["max_min_ratio"] = var["max"] / var["min"]
print(var.sort_values("max_min_ratio", ascending=False).head(20))


(1985253, 21)
         log_charge   log_tot_pay  log_mdcr_pay  charge_to_totpay  \
count  1.985253e+06  1.985253e+06  1.985253e+06      1.985253e+06   
mean   1.073052e+01  9.353351e+00  9.149499e+00      4.505747e+00   
std    8.194956e-01  6.673787e-01  7.149068e-01      2.397997e+00   
min    7.155722e+00  6.970815e+00  0.000000e+00      1.157366e-01   
25%    1.014432e+01  8.848730e+00  8.614650e+00      2.913890e+00   
50%    1.066690e+01  9.248240e+00  9.064299e+00      3.959446e+00   
75%    1.124971e+01  9.726953e+00  9.554897e+00      5.506535e+00   
max    1.615914e+01  1.354336e+01  1.352980e+01      3.915664e+01   

       totpay_to_mdcr  
count    1.985251e+06  
mean     1.238745e+00  
std      2.400056e-01  
min      1.000000e+00  
25%      1.115275e+00  
50%      1.194718e+00  
75%      1.304383e+00  
max      8.688334e+01  
         Year Rndrng_Prvdr_State_Abrvtn  Rndrng_Prvdr_CCN  DRG_Cd  \
1953428  2023                        PA            390164      18   
1807417  2

In [5]:
# pick common, high-volume DRGs
selected_drgs = [
"SEPTICEMIA OR SEVERE SEPSIS WITHOUT MV >96 HOURS WITH MCC",
"SEPTICEMIA OR SEVERE SEPSIS WITHOUT MV >96 HOURS WITHOUT MCC",
"HEART FAILURE AND SHOCK WITH MCC",
"SIMPLE PNEUMONIA AND PLEURISY WITH MCC",
"CHRONIC OBSTRUCTIVE PULMONARY DISEASE WITH MCC",
"RENAL FAILURE WITH CC",
"MISCELLANEOUS DISORDERS OF NUTRITION, METABOLISM, FLUIDS AND ELECTROLYTES WITHOUT MCC",
"MAJOR HIP AND KNEE JOINT REPLACEMENT OR REATTACHMENT OF LOWER EXTREMITY WITHOUT MCC",
"GASTROINTESTINAL HEMORRHAGE WITH CC",
"CARDIAC ARRHYTHMIA AND CONDUCTION DISORDERS WITH CC"
]

core = df[df["DRG_Desc"].isin(selected_drgs)].copy()

print(core.shape)
print(core["DRG_Desc"].value_counts())

# save core working dataset
core_file = os.path.join(save_path, "medicare_core_working.csv")
core.to_csv(core_file, index=False)


(260405, 21)
DRG_Desc
SEPTICEMIA OR SEVERE SEPSIS WITHOUT MV >96 HOURS WITH MCC                                30619
HEART FAILURE AND SHOCK WITH MCC                                                         29461
SEPTICEMIA OR SEVERE SEPSIS WITHOUT MV >96 HOURS WITHOUT MCC                             27366
SIMPLE PNEUMONIA AND PLEURISY WITH MCC                                                   27143
RENAL FAILURE WITH CC                                                                    25166
MAJOR HIP AND KNEE JOINT REPLACEMENT OR REATTACHMENT OF LOWER EXTREMITY WITHOUT MCC      25118
CHRONIC OBSTRUCTIVE PULMONARY DISEASE WITH MCC                                           24984
MISCELLANEOUS DISORDERS OF NUTRITION, METABOLISM, FLUIDS AND ELECTROLYTES WITHOUT MCC    24675
GASTROINTESTINAL HEMORRHAGE WITH CC                                                      24278
CARDIAC ARRHYTHMIA AND CONDUCTION DISORDERS WITH CC                                      21595
Name: count, dtype: int64


In [6]:
# variation by DRG
v = core.groupby("DRG_Desc")["Avg_Submtd_Cvrd_Chrg"].agg(
    min="min",
    median="median",
    max="max",
    count="count"
)

v["max_min_ratio"] = v["max"] / v["min"]
print(v.sort_values("max_min_ratio", ascending=False))


# variation by hospital within DRG
hosp_var = (
    core.groupby(["DRG_Desc", "Rndrng_Prvdr_CCN"])["Avg_Submtd_Cvrd_Chrg"]
    .median()
    .reset_index()
)

spread = hosp_var.groupby("DRG_Desc")["Avg_Submtd_Cvrd_Chrg"].agg(
    min="min",
    median="median",
    max="max"
)

spread["max_min_ratio"] = spread["max"] / spread["min"]
print(spread.sort_values("max_min_ratio", ascending=False))


                                                            min        median  \
DRG_Desc                                                                        
CHRONIC OBSTRUCTIVE PULMONARY DISEASE WITH MCC      4273.090909  31381.765883   
MISCELLANEOUS DISORDERS OF NUTRITION, METABOLIS...  3199.058824  21572.341463   
MAJOR HIP AND KNEE JOINT REPLACEMENT OR REATTAC...  4197.947368  57054.155872   
SEPTICEMIA OR SEVERE SEPSIS WITHOUT MV >96 HOUR...  4917.166667  29753.470355   
HEART FAILURE AND SHOCK WITH MCC                    5534.000000  35958.651724   
SEPTICEMIA OR SEVERE SEPSIS WITHOUT MV >96 HOUR...  6861.600000  49451.837838   
RENAL FAILURE WITH CC                               4608.000000  25523.755842   
SIMPLE PNEUMONIA AND PLEURISY WITH MCC              6631.545455  36989.666667   
GASTROINTESTINAL HEMORRHAGE WITH CC                 5911.000000  30264.417873   
CARDIAC ARRHYTHMIA AND CONDUCTION DISORDERS WIT...  4847.090909  23498.684211   

                           

In [7]:
# state-level median charges by DRG
state_med = (
    core.groupby(["DRG_Desc", "Rndrng_Prvdr_State_Abrvtn"])["Avg_Submtd_Cvrd_Chrg"]
    .median()
    .reset_index()
)

state_spread = state_med.groupby("DRG_Desc")["Avg_Submtd_Cvrd_Chrg"].agg(
    min="min",
    median="median",
    max="max"
)

state_spread["max_min_ratio"] = state_spread["max"] / state_spread["min"]
print(state_spread.sort_values("max_min_ratio", ascending=False))


# urban vs rural check
ruca = core.groupby("Rndrng_Prvdr_RUCA_Desc")["Avg_Submtd_Cvrd_Chrg"].median()
print(ruca.sort_values())


# save final core dataset used for modeling
final_core_file = os.path.join(save_path, "medicare_core_clean.csv")
core.to_csv(final_core_file, index=False)


                                                             min  \
DRG_Desc                                                           
CARDIAC ARRHYTHMIA AND CONDUCTION DISORDERS WIT...   8802.263158   
HEART FAILURE AND SHOCK WITH MCC                    14142.772727   
GASTROINTESTINAL HEMORRHAGE WITH CC                 11137.637123   
SEPTICEMIA OR SEVERE SEPSIS WITHOUT MV >96 HOUR...  20556.158369   
CHRONIC OBSTRUCTIVE PULMONARY DISEASE WITH MCC      12944.991072   
SIMPLE PNEUMONIA AND PLEURISY WITH MCC              14785.258065   
RENAL FAILURE WITH CC                               10644.680412   
SEPTICEMIA OR SEVERE SEPSIS WITHOUT MV >96 HOUR...  11884.222121   
MISCELLANEOUS DISORDERS OF NUTRITION, METABOLIS...   8992.022727   
MAJOR HIP AND KNEE JOINT REPLACEMENT OR REATTAC...  24028.375000   

                                                          median  \
DRG_Desc                                                           
CARDIAC ARRHYTHMIA AND CONDUCTION DISORDERS WIT

In [8]:
# standardize hospital id column name for merging later
core["CCN"] = core["Rndrng_Prvdr_CCN"].astype(str)

# keep only columns we actually need going forward
core = core[[
    "CCN",
    "Rndrng_Prvdr_Org_Name",
    "Rndrng_Prvdr_State_Abrvtn",
    "Rndrng_Prvdr_RUCA_Desc",
    "DRG_Cd",
    "DRG_Desc",
    "Tot_Dschrgs",
    "Avg_Submtd_Cvrd_Chrg",
    "Avg_Tot_Pymt_Amt",
    "Avg_Mdcr_Pymt_Amt",
    "Year",
    "log_charge",
    "log_tot_pay",
    "log_mdcr_pay"
]].copy()

print(core.shape)
print(core.head())

# save merge-ready dataset
merge_ready_file = os.path.join(save_path, "medicare_core_merge_ready.csv")
core.to_csv(merge_ready_file, index=False)


(260405, 14)
      CCN             Rndrng_Prvdr_Org_Name Rndrng_Prvdr_State_Abrvtn  \
30  10001  Southeast Alabama Medical Center                        AL   
33  10001  Southeast Alabama Medical Center                        AL   
66  10001  Southeast Alabama Medical Center                        AL   
74  10001  Southeast Alabama Medical Center                        AL   
87  10001  Southeast Alabama Medical Center                        AL   

                               Rndrng_Prvdr_RUCA_Desc  DRG_Cd  \
30  Metropolitan area core: primary flow within an...     190   
33  Metropolitan area core: primary flow within an...     193   
66  Metropolitan area core: primary flow within an...     291   
74  Metropolitan area core: primary flow within an...     309   
87  Metropolitan area core: primary flow within an...     378   

                                             DRG_Desc  Tot_Dschrgs  \
30     CHRONIC OBSTRUCTIVE PULMONARY DISEASE WITH MCC           64   
33             SI

In [10]:
import pandas as pd
import os

save_path = r"C:\Users\priva\OneDrive\Documents\Honors Project\Data"

core_file = os.path.join(save_path, "medicare_core_merge_ready.csv")
hosp_file = r"C:\Users\priva\OneDrive\Documents\Honors Project\Data\Hospital_General_Information.csv.xlsx"

core = pd.read_csv(core_file, dtype={"CCN": str})
hosp = pd.read_excel(hosp_file, dtype={"Facility ID": str})

# rename for merge
hosp = hosp.rename(columns={"Facility ID": "CCN"})

# keep needed columns
hosp = hosp[[
    "CCN",
    "Hospital Type",
    "Hospital Ownership",
    "Emergency Services",
    "Hospital overall rating",
    "MORT Group Measure Count",
    "Safety Group Measure Count",
    "READM Group Measure Count",
    "Pt Exp Group Measure Count"
]].copy()

merged = core.merge(hosp, on="CCN", how="left")

print(merged.shape)
print("missing hospital matches:", merged["Hospital Type"].isna().mean())

# save merged dataset
merged_file = os.path.join(save_path, "medicare_core_with_hospital_info.csv")
merged.to_csv(merged_file, index=False)


(260405, 22)
missing hospital matches: 0.05267948004070582


In [11]:
import pandas as pd
import os

save_path = r"C:\Users\priva\OneDrive\Documents\Honors Project\Data"

merged_file = os.path.join(save_path, "medicare_core_with_hospital_info.csv")
merged = pd.read_csv(merged_file, dtype={"CCN": str})

missing = merged[merged["Hospital Type"].isna()][["CCN", "Rndrng_Prvdr_Org_Name", "Rndrng_Prvdr_State_Abrvtn"]].drop_duplicates()

print(missing.shape)
print(missing.head(25))

missing_file = os.path.join(save_path, "missing_hospital_matches.csv")
missing.to_csv(missing_file, index=False)

print("saved:", missing_file)


(582, 3)
        CCN                   Rndrng_Prvdr_Org_Name Rndrng_Prvdr_State_Abrvtn
102   10025      George H. Lanier Memorial Hospital                        AL
122   10032                        Wedowee Hospital                        AL
161   10038          Stringfellow Memorial Hospital                        AL
209   10047                Georgiana Medical Center                        AL
221   10054  Decatur Morgan Hospital-Parkway Campus                        AL
280   10069                  Medical Center Barbour                        AL
415   10109           Pickens County Medical Center                        AL
501   10146             Jacksonville Medical Center                        AL
613   30001                       Maryvale Hospital                        AZ
739   30033          Payson Regional Medical Center                        AZ
835   30067                La Paz Regional Hospital                        AZ
837   30068       Mt Graham Regional Medical Center    

In [12]:
# keep all rows, but create a flag for match
merged["has_hosp_info"] = (~merged["Hospital Type"].isna()).astype(int)

print(merged["has_hosp_info"].value_counts(normalize=True))


has_hosp_info
1    0.947321
0    0.052679
Name: proportion, dtype: float64


In [13]:
import os

save_path = r"C:\Users\priva\OneDrive\Documents\Honors Project\Data"
final_file = os.path.join(save_path, "thesis_working_dataset.csv")

merged.to_csv(final_file, index=False)

print("saved:", final_file)


saved: C:\Users\priva\OneDrive\Documents\Honors Project\Data\thesis_working_dataset.csv


In [14]:
# basic charge vs medicare relationship
print(merged[["Avg_Submtd_Cvrd_Chrg", "Avg_Mdcr_Pymt_Amt"]].corr())

# by hospital type
print(
    merged.groupby("Hospital Type")["Avg_Submtd_Cvrd_Chrg"]
    .median()
    .sort_values(ascending=False)
)


                      Avg_Submtd_Cvrd_Chrg  Avg_Mdcr_Pymt_Amt
Avg_Submtd_Cvrd_Chrg              1.000000           0.436976
Avg_Mdcr_Pymt_Amt                 0.436976           1.000000
Hospital Type
Acute Care Hospitals    33639.40678
Name: Avg_Submtd_Cvrd_Chrg, dtype: float64


In [15]:
import numpy as np

# log outcome for modeling
merged["log_charge"] = np.log(merged["Avg_Submtd_Cvrd_Chrg"])
merged["log_mdcr_pay"] = np.log(merged["Avg_Mdcr_Pymt_Amt"].clip(lower=1))

# payment gap vs medicare
merged["charge_to_mdcr"] = (
    merged["Avg_Submtd_Cvrd_Chrg"] /
    merged["Avg_Mdcr_Pymt_Amt"].replace(0, np.nan)
)

print(merged[["log_charge","log_mdcr_pay","charge_to_mdcr"]].describe())


          log_charge   log_mdcr_pay  charge_to_mdcr
count  260405.000000  260405.000000   260405.000000
mean       10.441666       8.911136        5.321056
std         0.607685       0.424222        2.973299
min         8.070612       6.672608        0.141882
25%        10.012440       8.595299        3.325674
50%        10.418678       8.897436        4.642266
75%        10.852220       9.211442        6.568578
max        13.630182      11.997384       65.657964


In [16]:
# small sample for sharing
sample = merged.sample(n=10000, random_state=1)

sample_file = os.path.join(save_path, "thesis_dataset_sample.csv")
sample.to_csv(sample_file, index=False)

print("sample saved:", sample_file)


sample saved: C:\Users\priva\OneDrive\Documents\Honors Project\Data\thesis_dataset_sample.csv
